# Mensageiros

*AMQP* (Advanced Message Queuing Protocol) é um protocolo aberto para comunicação entre sistemas distribuídos.

Ele define como mensagens são publicadas, roteadas, enfileiradas e entregues.

*RabbitMQ* é um dos brokers mais usados que implementa AMQP.

## Principais Conceitos

*Producer*: Envia mensagens.

*Exchange*: Recebe as mensagens dos producers e as roteia para as filas/queues.

*Queue*: Filas de mensagem, armazenando-as até serem consumidas.

*Binding*: Ligação entre Exchange e Queue.

*Consumer*: Recebe/Lê as mensagens.

*Connection*: Conexão TCP entre o cliente (producer ou consumer) com o Broker (e.g., RabbitMQ).

*Channel*: Túnel lógico dentro da conexão (evita abrir várias conexões).

## Arquitetura

![Arquitetura](./images/arquitetura.png)

## Tipos de Exchange

Direct → roteia por chave exata.
Exemplo: routing key orange só vai para a fila ligada com orange.

Fanout → broadcast para todas as filas ligadas.
Exemplo: log ou evento que todo mundo precisa receber.

Topic → roteamento por padrão (wildcards).
Exemplo: user.signup, order.*, *.error.

Headers → roteamento por cabeçalhos (não usa routing key).

## Ciclo da Mensagem

Producer --> Exchange --> Queue --> Consumer

## Producer em Python

In [8]:
import pika

# Conexão com RabbitMQ
conn = pika.BlockingConnection(pika.ConnectionParameters("localhost"))
ch = conn.channel()

# Declara exchange do tipo direct
ch.exchange_declare(exchange="ex.direct", exchange_type="direct")

# Envia mensagem com routing key "hello"
msg = "Olá AMQP!"
ch.basic_publish(exchange="ex.direct", routing_key="hello", body=msg)

print("Producer enviou:", msg)
conn.close()

Producer enviou: Olá AMQP!


## Consumer em Python

In [ ]:
import pika

def callback(ch, method, properties, body):
    print("Consumer recebeu:", body.decode())

conn = pika.BlockingConnection(pika.ConnectionParameters("localhost"))
ch = conn.channel()

# Cria fila e liga ao exchange
ch.queue_declare(queue="fila_hello")
ch.queue_bind(exchange="ex.direct", queue="fila_hello", routing_key="hello")

# Consome
ch.basic_consume(queue="fila_hello", on_message_callback=callback, auto_ack=True)

print("Aguardando mensagens...")
ch.start_consuming()

## Broadcast

In [ ]:
# Producer
ch.exchange_declare(exchange="ex.fanout", exchange_type="fanout")
ch.basic_publish(exchange="ex.fanout", routing_key="", body="Mensagem para todos")

# Consumer
result = ch.queue_declare(queue="", exclusive=True)
queue_name = result.method.queue
ch.queue_bind(exchange="ex.fanout", queue=queue_name)

## Autenticação

In [ ]:
import pika

conn = pika.BlockingConnection(
    pika.ConnectionParameters(
        host="localhost",
        port=5672,
        virtual_host="/app",
        credentials=pika.PlainCredentials("meuusuario", "minhasenha")
    )
)
ch = conn.channel()
print("Conectado com sucesso!")


## Servidor RabbitMQ

Para acessar, usar `http://localhost:15672` com usuário e senha `guest`.